### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.3.17 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


<a name="Data"></a>
### Data Prep (For Symptoms dataset)

In [ ]:
import pandas as pd
import random
from datasets import Dataset

#########################################################
# LOAD DATASET
#########################################################

df = pd.read_csv("/content/cleaned_symptoms_dataset.csv")

#########################################################
# CLEAN DATA
#########################################################

df["Symptoms"] = df["Symptoms"].str.lower().str.strip()
df["Disease"] = df["Disease"].str.lower().str.strip()
df["Speciality"] = df["Speciality"].str.lower().str.strip()

# Remove missing values
df = df.dropna(subset=["Symptoms", "Disease", "Speciality", "Symptom_Count"])

#########################################################
# CONFIDENCE FUNCTION
#########################################################

def generate_confidence(symptom_count):
    if symptom_count >= 6:
        base = 0.92
    elif symptom_count == 5:
        base = 0.88
    elif symptom_count == 4:
        base = 0.82
    elif symptom_count == 3:
        base = 0.75
    else:
        base = 0.65

    noise = random.uniform(-0.05, 0.05)
    confidence = min(max(base + noise, 0.6), 0.95)

    return round(confidence, 2)

#########################################################
# ALPACA PROMPT TEMPLATE
#########################################################

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

#########################################################
# TOKENIZER EOS (REQUIRED)
#########################################################

EOS_TOKEN = tokenizer.eos_token

#########################################################
# FORMAT FUNCTION
#########################################################

def formatting_prompts_func(examples):
    symptoms = examples["Symptoms"]
    diseases = examples["Disease"]
    specialists = examples["Speciality"]
    counts = examples["Symptom_Count"]

    texts = []

    for sym, dis, spec, count in zip(symptoms, diseases, specialists, counts):

        # Instruction variations
        instruction = random.choice([
            "Identify the most likely disease based on the symptoms and recommend the appropriate medical specialist. Also include a confidence score.",
            "Analyze the symptoms, determine the disease, suggest a specialist, and provide confidence level.",
            "Based on symptoms, predict the illness, recommend a doctor, and give a confidence score."
        ])

        input_text = f"Symptoms: {sym}"

        confidence = generate_confidence(count)

        # Output variations
        output_text = random.choice([
            f"Possible Disease: {dis}\nRecommended Specialist: {spec}\nConfidence: {confidence}",
            f"The patient may have {dis}. Suggested specialist: {spec}. Confidence score: {confidence}",
            f"Likely condition: {dis}. Consult: {spec}. Confidence: {confidence}"
        ])

        full_text = alpaca_prompt.format(instruction, input_text, output_text) + EOS_TOKEN
        texts.append(full_text)

    return {"text": texts}

#########################################################
# CONVERT TO HF DATASET
#########################################################

dataset = Dataset.from_pandas(df)

#########################################################
# APPLY FORMATTING
#########################################################

dataset = dataset.map(formatting_prompts_func, batched=True)

#########################################################
# SAVE FINAL DATASET (OPTIONAL)
#########################################################

dataset.to_csv("alpaca_symptomsTraining_dataset.csv")

#########################################################
# PREVIEW
#########################################################

print(dataset[0]["text"])
print("\n✅ Dataset ready for fine-tuning!")

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Based on the symptoms, determine the disease and suggest the correct specialist.

### Input:
Symptoms: appetite loss, runny nose, shortness of breath

### Response:
Disease: depression
Speciality: psychiatry<|end_of_text|>


### Data Prep 2 (Facilities dataset)


In [7]:
import pandas as pd
from datasets import Dataset
import random

#########################################################
# 1️⃣ LOAD DATASETS
#########################################################

# Patient dataset
patient_df = pd.read_csv("Synthetic Data files\synthetic-data-kit\cleaned_symptoms_dataset_noencode.csv")

# OSM healthcare facilities dataset
facility_df = pd.read_csv("Synthetic Data files\synthetic-data-kit\dataset_facilities_CLEANED.csv")

#########################################################
# 2️⃣ CLEAN DATA
#########################################################

# Normalize text
patient_df["Symptoms"] = patient_df["Symptoms"].str.lower().str.strip()
patient_df["Disease"] = patient_df["Disease"].str.lower().str.strip()
patient_df["Speciality"] = patient_df["Speciality"].str.lower().str.strip()

facility_df["healthcare:speciality"] = facility_df["healthcare:speciality"].fillna("general")
facility_df["healthcare:speciality"] = facility_df["healthcare:speciality"].str.lower().str.strip()

#########################################################
# 3️⃣ MATCH SPECIALITY → FACILITIES
#########################################################

def get_facilities_by_speciality(speciality):
    matches = facility_df[facility_df["healthcare:speciality"] == speciality]

    if matches.empty:
        matches = facility_df[facility_df["healthcare:speciality"] == "general"]

    # Sample up to 2 facilities
    matches = matches.sample(min(len(matches), 2))

    facilities = []
    for _, row in matches.iterrows():
        name = row["name"]
        location = row["addr:city"]
        facilities.append(f"{name} ({location})")

    return facilities

#########################################################
# 4️⃣ CREATE ALPACA DATASET
#########################################################

alpaca_data = []

for _, row in patient_df.iterrows():

    instruction = (
        "Given a patient's symptoms, identify the possible disease, "
        "recommend the appropriate medical specialty, and suggest suitable healthcare facilities."
    )

    input_text = f"Patient symptoms: {row['Symptoms']}"

    disease = row["Disease"]
    speciality = row["Speciality"]

    facilities = get_facilities_by_speciality(speciality)

    facilities_text = ", ".join(facilities)

    output_text = (
        f"Based on the symptoms, the patient may have {disease}. "
        f"It is recommended to consult a specialist in {speciality}. "
        f"Suggested healthcare facilities include: {facilities_text}."
    )

    alpaca_data.append({
        "instruction": instruction,
        "input": input_text,
        "output": output_text
    })

#########################################################
# 5️⃣ CONVERT TO DATASET
#########################################################

alpaca_df = pd.DataFrame(alpaca_data)

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(example):
    return {
        "text": alpaca_prompt.format(
            example["instruction"],
            example["input"],
            example["output"]
        )
    }

dataset = Dataset.from_pandas(alpaca_df)
dataset = dataset.map(formatting_prompts_func)

#########################################################
# 6️⃣ TEST OUTPUT
#########################################################

print(dataset[0]["text"])

FileNotFoundError: [Errno 2] No such file or directory: 'Synthetic Data files\\synthetic-data-kit\\cleaned_symptoms_dataset_noencode.csv'

### Test 3 (Using symptoms to determine specialist)

In [ ]:
import pandas as pd
from datasets import Dataset
import ast

#########################################################
# 1️⃣ LOAD PREPROCESSED DATASETS
#########################################################
patient_df = pd.read_csv("cleaned_symptoms_all_bodyparts.csv")
facility_df = pd.read_csv("dataset_facilities_CLEANED.csv")

# Ensure healthcare:speciality is lowercase for matching
facility_df["healthcare:speciality"] = facility_df["healthcare:speciality"].fillna("general")
facility_df["healthcare:speciality"] = facility_df["healthcare:speciality"].str.lower().str.strip()

#########################################################
# 2️⃣ FACILITY MATCHER FUNCTION
#########################################################
def get_facilities_for_specialty(specialty, top_k=3):
    """Return a list of facilities matching the specialty"""
    specialty = str(specialty).lower().strip()
    matches = facility_df[facility_df["healthcare:speciality"] == specialty]
    
    if matches.empty:
        matches = facility_df[facility_df["healthcare:speciality"] == "general"]
    if matches.empty:
        return ["No facility found"]
    
    matches = matches.head(top_k)
    return [f"{row['name']} ({row['addr:city']})" for _, row in matches.iterrows()]

#########################################################
# 3️⃣ CREATE ALPACA DATASET
#########################################################

alpaca_data = []

# Ensure Symptoms_List is a list
patient_df["Symptoms_List"] = patient_df["Symptoms_List"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

for _, row in patient_df.iterrows():
    symptoms = row["Symptoms_List"]
    body_parts = row["Body_Part_Breakdown"]
    speciality = row["Speciality"]
    facilities = get_facilities_for_specialty(speciality)

    instruction = (
        "Given a patient's symptoms, determine the body parts affected, "
        "recommend the appropriate medical specialty, and suggest suitable healthcare facilities."
    )
    input_text = f"Patient symptoms: {symptoms}"
    output_text = (
        f"Body parts breakdown: {body_parts}\n"
        f"Recommended specialty: {speciality}\n"
        f"Suggested facilities: {facilities}"
    )

    alpaca_data.append({
        "instruction": instruction,
        "input": input_text,
        "output": output_text
    })

#########################################################
# 4️⃣ CONVERT TO HF DATASET AND FORMAT PROMPT
#########################################################

alpaca_df = pd.DataFrame(alpaca_data)

alpaca_prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input_text}

### Response:
{output_text}"""

def format_to_alpaca(example):
    return {
        "text": alpaca_prompt_template.format(
            instruction=example["instruction"],
            input_text=example["input"],
            output_text=example["output"]
        )
    }

dataset = Dataset.from_pandas(alpaca_df)
dataset = dataset.map(format_to_alpaca)

#########################################################
# 5️⃣ TEST OUTPUT
#########################################################

# Show first two examples
print(dataset[0]["text"])
print("\n------------------\n")
print(dataset[1]["text"])

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Given a patient's symptoms, determine the body parts affected, recommend the appropriate medical specialty, and suggest suitable healthcare facilities.

### Input:
Patient symptoms: ['fever', 'back pain', 'shortness of breath']

### Response:
Body parts breakdown: ['chest', 'back', 'whole body']
Recommended specialty: pulmonology
Suggested facilities: ['Poliklinik Simpang (Taiping)', 'Klinik Reddy (Puchong)', 'Klinik Selva (Puchong)']

------------------

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Given a patient's symptoms, determine the body parts affected, recommend the appropriate medical specialty, and suggest suitable healthcare facilities.

### Input:
Patient symptoms: ['inso

### Test 4 (using symptoms where the context of disease is broken down into its specialist and body part, emergency level of disease and facility type based on emergency level)

In [21]:
import pandas as pd
from datasets import Dataset

# --------------------------
# 1️⃣ Load your datasets
# --------------------------
patients_df = pd.read_csv("cleaned_symptoms_dataset_bodyparts_by_disease.csv")
facilities_df = pd.read_csv("healthcare_facility_cleaned_v3.csv")

# --------------------------
# 2️⃣ Helper function to pick facilities
# --------------------------
def recommend_facilities(facility_type, top_n=3):
    df = facilities_df[facilities_df['healthcare'].str.lower() == facility_type.lower()]
    recommended = df['name'].tolist()[:top_n]
    return ", ".join(recommended) if recommended else "No suitable facility found"

# --------------------------
# 3️⃣ Alpaca-style template in instruction/input/output format
# --------------------------
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token  # Make sure your tokenizer is loaded

# --------------------------
# 4️⃣ Format the dataset
# --------------------------
def formatting_healthcare_prompts(df):
    texts = []
    instructions = []
    inputs_ = []
    outputs = []

    for _, row in df.iterrows():
        # Instruction: describe the task
        instruction = "Given patient symptoms, determine the affected body parts, recommend the appropriate medical specialty, and suggest suitable healthcare facilities based on emergency level."

        # Input: the patient symptoms
        input_text = row['Symptoms']

        # Output: ground truth including body parts, specialty, and recommended facilities
        body_parts = row['Affected_BodyPart']
        speciality = row['Speciality']
        facility_type = row['Recommended_Facility_Type']
        recommended_facilities = recommend_facilities(facility_type)

        output_text = (
            f"Affected Body Parts: {body_parts}. "
            f"Recommended Specialty: {speciality}. "
            f"Suggested Facilities: {recommended_facilities}."
        )

        # Append to lists
        instructions.append(instruction)
        inputs_.append(input_text)
        outputs.append(output_text)

        # Combine into the final Alpaca-style prompt with EOS token
        texts.append(alpaca_prompt.format(instruction, input_text, output_text) + EOS_TOKEN)

    return {
        "instruction": instructions,
        "input": inputs_,
        "output": outputs,
        "text": texts
    }

# --------------------------
# 5️⃣ Convert to HuggingFace Dataset
# --------------------------
formatted_data = formatting_healthcare_prompts(patients_df)
alpaca_healthcare_dataset = Dataset.from_dict(formatted_data)

# --------------------------
# 6️⃣ Inspect a few samples
# --------------------------
print(alpaca_healthcare_dataset["text"])

Column(['Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nGiven patient symptoms, determine the affected body parts, recommend the appropriate medical specialty, and suggest suitable healthcare facilities based on emergency level.\n\n### Input:\nfever, back pain, shortness of breath\n\n### Response:\nAffected Body Parts: skin, respiratory system. Recommended Specialty: immunology. Suggested Facilities: Guardian, AA Pharmacy, Big Pharmacy.<|end_of_text|>', 'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nGiven patient symptoms, determine the affected body parts, recommend the appropriate medical specialty, and suggest suitable healthcare facilities based on emergency level.\n\n### Input:\ninsomnia, back pain, weight loss\n\n### Re

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!!

In [19]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = alpaca_healthcare_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/25000 [00:00<?, ? examples/s]

In [27]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
6.705 GB of memory reserved.


### Start Training the Model!

In [20]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 25,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,3.425700
2,3.424600
3,3.421900
4,3.123900
5,2.829200
6,2.469000
7,2.066500
8,1.692600
9,1.263300
10,0.914800


Unsloth: Will smartly offload gradients to save VRAM!


In [34]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

NameError: name 'start_gpu_memory' is not defined

<a name="Inference"></a>
### Inference
Time to run the model!

In [ ]:
# 1️⃣ Alpaca-style prompt template
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# 2️⃣ Example: user-provided symptoms
user_symptoms = "I got stomach pain"

# 3️⃣ Fill the prompt with your instruction and symptoms
prompt = alpaca_prompt.format(
    "Given the symptoms, determine the disease and the specialist to consult.",  # instruction
    f"Symptoms: {user_symptoms}",  # input
    ""  # leave output blank for generation
)

# 4️⃣ Tokenize and move to GPU
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

# 5️⃣ Generate output
outputs = model.generate(
    **inputs,
    max_new_tokens=64,
    use_cache=True,
    do_sample=True,   # optional: makes generation more diverse
    temperature=0.7,  # optional: controls creativity
    top_p=0.9         # optional: nucleus sampling
)

# 6️⃣ Decode and remove the prompt from output
generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
model_response = generated_text[len(prompt):].strip()

# 7️⃣ Print results
print("Symptoms:", user_symptoms)
print("Model Prediction:\n", model_response)

Symptoms: I got stomach pain
Model Prediction:
 Disease: depression
Speciality: psychiatry


### Inference 2 (facilities dataset)

In [ ]:
# ------------------------------
# Updated Alpaca Prompt
# ------------------------------
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Given the patient's symptoms, identify the most likely disease, recommend the appropriate medical specialty, suggest suitable healthcare facilities, and include a confidence score.

### Input:
Patient symptoms: {}

### Response:
{}"""

# ------------------------------
# Load facility dataset (IMPORTANT)
# ------------------------------
import pandas as pd

facility_df = pd.read_csv("dataset_facilities_CLEANED.csv")

facility_df["healthcare:speciality"] = facility_df["healthcare:speciality"].fillna("general")
facility_df["healthcare:speciality"] = facility_df["healthcare:speciality"].str.lower().str.strip()

# ------------------------------
# Facility matcher
# ------------------------------
import random

def get_facilities_by_speciality(speciality, top_k=2):
    matches = facility_df[facility_df["healthcare:speciality"] == speciality]

    if matches.empty:
        matches = facility_df[facility_df["healthcare:speciality"] == "general"]

    matches = matches.sample(min(len(matches), top_k))

    facilities = []
    for _, row in matches.iterrows():
        facilities.append(f"{row['name']} ({row['addr:city']})")

    return facilities

# ------------------------------
# Extract speciality from model output (simple parser)
# ------------------------------
import re

def extract_speciality(text):
    match = re.search(r"specialist in (.+?)[\.,]", text.lower())
    if match:
        return match.group(1).strip()
    return "general"

# ------------------------------
# Inference function (UPDATED)
# ------------------------------
def predict_disease(symptoms_input, model, tokenizer, max_tokens=128):

    prompt_text = alpaca_prompt.format(symptoms_input, "")

    inputs = tokenizer([prompt_text], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
        use_cache=True
    )

    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    completion = generated_text[len(prompt_text):].strip()

    # ------------------------------
    # NEW: Add facility recommendation dynamically
    # ------------------------------
    speciality = extract_speciality(completion)
    facilities = get_facilities_by_speciality(speciality)

    facilities_text = ", ".join(facilities)

    final_response = (
        f"{completion}\n\n"
        f"Recommended healthcare facilities: {facilities_text}"
    )

    return final_response

# ------------------------------
# Example usage
# ------------------------------
user_symptoms = "fever, back pain, shortness of breath"

prediction = predict_disease(user_symptoms, model, tokenizer)

print("=== User Symptoms ===")
print(user_symptoms)

print("\n=== Model Prediction ===")
print(prediction)

Symptoms: I got back pain and shortness of breath
Model Prediction:
 Based on your symptoms, you may be experiencing Chronic kidney disease. It is recommended to consult a Nephrology specialist. A suitable healthcare facility would be Local Nephrology Clinic (clinic).://M

### Input:
I'm experiencing dizziness, weight gain, rash, nausea, appetite loss, nan
--------------------------------------------------------------------------------


### Prep 3 (using symptoms to match to specialist)

In [ ]:
# --------------------------
# Interactive Inference Test
# --------------------------

# Replace this with any symptoms you want to test
user_input = input("fever, back pain, shortness of breath\n")

# Prepare Alpaca-style prompt
test_prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Given patient symptoms, determine the affected body parts, recommend the appropriate medical specialty, and suggest suitable healthcare facilities based on emergency level.

### Symptoms:
{user_input}

### Response:
<EOS>"""  # Use your model's EOS token

# Tokenize and send to GPU
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

# Generate output
outputs = model.generate(
    **inputs,
    max_new_tokens=64,
    use_cache=True,
    eos_token_id=tokenizer.convert_tokens_to_ids("<EOS>")  # Adjust if your EOS is different
)

# Decode and display
decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print("\nGenerated Response:\n", decoded_output)


Generated Response:
 Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Given patient symptoms, determine the affected body parts, recommend the appropriate medical specialty, and suggest suitable healthcare facilities based on emergency level.

### Symptoms:


### Response:
<EOS> affected body parts: heart. Recommended specialty: cardiology. Suggested facilities: Nazirin Skin Clinic, Klinik Dr Agnes Lim, Lau Kee Oon Specialist Clinic.:// affected body parts: brain. Recommended specialty: neurology. Suggested facilities: Majestic Maxim Clinic, Lau Kee Oon


# Model download

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("llama_lora")  # Local saving
tokenizer.save_pretrained("llama_lora")
# model.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "llama_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# alpaca_prompt = You MUST copy from above!

inputs = tokenizer(
[
    alpaca_prompt.format(
        "What is a famous tall tower in Paris?", # instruction
        "", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
What is a famous tall tower in Paris?

### Input:


### Response:
One of the most famous and iconic tall towers in Paris is the Eiffel Tower. Standing at 324 meters (1,063 feet) tall, this wrought iron tower is a symbol of the city and a must-see attraction for tourists from all over the world.<|end_of_text|>

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("llama_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/llama_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("llama_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/llama_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("llama_lora")
    tokenizer.save_pretrained("llama_lora")
if False:
    model.push_to_hub("HF_USERNAME/llama_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/llama_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("llama_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/llama_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)</b>
</div>